# Log Anaylsis and SIEM Tool

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import re
import json
import sqlite3
import hashlib
import time
from datetime import datetime, timedelta
from collections import defaultdict, Counter
import threading
import queue
import os
import gzip
import ipaddress
import csv
import pickle

class LogAnalyzer:
    def __init__(self, db_path="security_logs.db"):
        self.db_path = db_path
        self.setup_database()
        self.alerts = []
        self.patterns = self.load_threat_patterns()
        self.whitelist = set()
        self.blacklist = set()
        self.geo_cache = {}

    def setup_database(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT,
                source_ip TEXT,
                source_port INTEGER,
                dest_ip TEXT,
                dest_port INTEGER,
                protocol TEXT,
                event_type TEXT,
                severity TEXT,
                message TEXT,
                parsed_data TEXT,
                hash TEXT
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS alerts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT,
                alert_type TEXT,
                severity TEXT,
                source_ip TEXT,
                destination TEXT,
                description TEXT,
                evidence TEXT,
                status TEXT DEFAULT 'open'
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS threat_intel (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                indicator TEXT UNIQUE,
                type TEXT,
                severity TEXT,
                description TEXT,
                source TEXT,
                timestamp TEXT
            )
        ''')

        conn.commit()
        conn.close()

    def load_threat_patterns(self):
        return {
            'brute_force': [
                r'failed login.*from.*(\d+\.\d+\.\d+\.\d+)',
                r'authentication failure.*rhost=(\d+\.\d+\.\d+\.\d+)',
                r'invalid user.*from (\d+\.\d+\.\d+\.\d+)'
            ],
            'sql_injection': [
                r'union.*select',
                r'or.*1=1',
                r'drop.*table',
                r'exec.*xp_cmdshell',
                r'script.*alert'
            ],
            'xss_attempts': [
                r'<script.*>',
                r'javascript:',
                r'onerror=',
                r'onload=',
                r'document\.cookie'
            ],
            'directory_traversal': [
                r'\.\.\/.*\.\.\/.*\.\.\/',
                r'\.\.\\.*\.\.\\.*\.\.\\',
                r'\/etc\/passwd',
                r'\/windows\/system32'
            ],
            'malware_communication': [
                r'POST.*\/[a-zA-Z0-9]{32}',
                r'User-Agent:.*bot',
                r'base64.*[A-Za-z0-9+\/]{100,}',
                r'cmd\.exe',
                r'powershell.*-enc'
            ],
            'port_scan': [
                r'connection.*refused.*(\d+\.\d+\.\d+\.\d+)',
                r'syn.*flood',
                r'port.*scan'
            ],
            'privilege_escalation': [
                r'sudo.*root',
                r'su.*root',
                r'privilege.*escalation',
                r'setuid.*0'
            ]
        }

    def parse_apache_log(self, log_line):
        apache_pattern = r'(\d+\.\d+\.\d+\.\d+) - - \[(.*?)\] "(.*?)" (\d+) (\d+) "(.*?)" "(.*?)"'
        match = re.match(apache_pattern, log_line)

        if match:
            return {
                'source_ip': match.group(1),
                'timestamp': match.group(2),
                'request': match.group(3),
                'status_code': int(match.group(4)),
                'response_size': int(match.group(5)),
                'referer': match.group(6),
                'user_agent': match.group(7)
            }
        return None

    def parse_nginx_log(self, log_line):
        nginx_pattern = r'(\d+\.\d+\.\d+\.\d+) - - \[(.*?)\] "(.*?)" (\d+) (\d+) "(.*?)" "(.*?)"'
        match = re.match(nginx_pattern, log_line)

        if match:
            return {
                'source_ip': match.group(1),
                'timestamp': match.group(2),
                'request': match.group(3),
                'status_code': int(match.group(4)),
                'response_size': int(match.group(5)),
                'referer': match.group(6),
                'user_agent': match.group(7)
            }
        return None

    def parse_ssh_log(self, log_line):
        ssh_patterns = [
            r'(\w+\s+\d+\s+\d+:\d+:\d+).*Failed password for.*from (\d+\.\d+\.\d+\.\d+)',
            r'(\w+\s+\d+\s+\d+:\d+:\d+).*Invalid user.*from (\d+\.\d+\.\d+\.\d+)',
            r'(\w+\s+\d+\s+\d+:\d+:\d+).*Accepted.*from (\d+\.\d+\.\d+\.\d+)'
        ]

        for pattern in ssh_patterns:
            match = re.search(pattern, log_line)
            if match:
                return {
                    'timestamp': match.group(1),
                    'source_ip': match.group(2),
                    'event_type': 'ssh_auth',
                    'message': log_line
                }
        return None

    def parse_firewall_log(self, log_line):
        firewall_pattern = r'(\d+\.\d+\.\d+\.\d+):(\d+) -> (\d+\.\d+\.\d+\.\d+):(\d+) \[(\w+)\] (ACCEPT|DENY|DROP)'
        match = re.match(firewall_pattern, log_line)

        if match:
            return {
                'source_ip': match.group(1),
                'source_port': int(match.group(2)),
                'dest_ip': match.group(3),
                'dest_port': int(match.group(4)),
                'protocol': match.group(5),
                'action': match.group(6)
            }
        return None

    def detect_threats(self, log_entry):
        threats = []
        message = log_entry.get('message', '') + ' ' + log_entry.get('request', '')

        for threat_type, patterns in self.patterns.items():
            for pattern in patterns:
                if re.search(pattern, message, re.IGNORECASE):
                    threats.append({
                        'type': threat_type,
                        'pattern': pattern,
                        'severity': self.get_threat_severity(threat_type),
                        'evidence': message[:200]
                    })

        return threats

    def get_threat_severity(self, threat_type):
        severity_map = {
            'brute_force': 'medium',
            'sql_injection': 'high',
            'xss_attempts': 'medium',
            'directory_traversal': 'high',
            'malware_communication': 'critical',
            'port_scan': 'low',
            'privilege_escalation': 'critical'
        }
        return severity_map.get(threat_type, 'medium')

    def analyze_brute_force(self, time_window=300):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        query = '''
            SELECT source_ip, COUNT(*) as attempts
            FROM logs
            WHERE event_type = 'ssh_auth'
            AND message LIKE '%Failed%'
            AND timestamp > datetime('now', '-{} seconds')
            GROUP BY source_ip
            HAVING attempts > 5
        '''.format(time_window)

        cursor.execute(query)
        results = cursor.fetchall()
        conn.close()

        alerts = []
        for ip, attempts in results:
            alerts.append({
                'type': 'brute_force_attack',
                'source_ip': ip,
                'attempts': attempts,
                'severity': 'high' if attempts > 20 else 'medium',
                'description': f'Brute force attack detected from {ip} with {attempts} failed attempts'
            })

        return alerts

    def analyze_anomalies(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        anomalies = []

        cursor.execute('''
            SELECT source_ip, COUNT(*) as request_count
            FROM logs
            WHERE timestamp > datetime('now', '-1 hour')
            GROUP BY source_ip
            HAVING request_count > 1000
        ''')

        for ip, count in cursor.fetchall():
            anomalies.append({
                'type': 'high_request_volume',
                'source_ip': ip,
                'request_count': count,
                'severity': 'medium',
                'description': f'Unusual request volume from {ip}: {count} requests in last hour'
            })

        cursor.execute('''
            SELECT dest_port, COUNT(*) as connection_count
            FROM logs
            WHERE timestamp > datetime('now', '-1 hour')
            GROUP BY dest_port
            HAVING connection_count > 100
        ''')

        for port, count in cursor.fetchall():
            anomalies.append({
                'type': 'port_scanning',
                'destination_port': port,
                'connection_count': count,
                'severity': 'medium',
                'description': f'Potential port scanning on port {port}: {count} connections'
            })

        conn.close()
        return anomalies

    def correlate_events(self, time_window=600):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        correlations = []

        cursor.execute('''
            SELECT source_ip, GROUP_CONCAT(event_type) as events
            FROM logs
            WHERE timestamp > datetime('now', '-{} seconds')
            GROUP BY source_ip
            HAVING COUNT(DISTINCT event_type) > 2
        '''.format(time_window))

        for ip, events in cursor.fetchall():
            event_types = events.split(',')
            if len(set(event_types)) > 2:
                correlations.append({
                    'type': 'multi_stage_attack',
                    'source_ip': ip,
                    'events': list(set(event_types)),
                    'severity': 'high',
                    'description': f'Multi-stage attack pattern from {ip}: {", ".join(set(event_types))}'
                })

        conn.close()
        return correlations

    def generate_threat_intel(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('''
            SELECT source_ip, COUNT(*) as alert_count
            FROM alerts
            WHERE severity IN ('high', 'critical')
            AND timestamp > datetime('now', '-7 days')
            GROUP BY source_ip
            HAVING alert_count > 3
        ''')

        threat_ips = []
        for ip, count in cursor.fetchall():
            cursor.execute('''
                INSERT OR REPLACE INTO threat_intel
                (indicator, type, severity, description, source, timestamp)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (ip, 'ip_address', 'high', f'Malicious IP with {count} alerts', 'internal_analysis', datetime.now().isoformat()))

            threat_ips.append(ip)

        conn.commit()
        conn.close()

        return threat_ips

    def process_log_file(self, file_path, log_type='auto'):
        processed_count = 0

        try:
            if file_path.endswith('.gz'):
                file_handle = gzip.open(file_path, 'rt')
            else:
                file_handle = open(file_path, 'r')

            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()

            for line in file_handle:
                line = line.strip()
                if not line:
                    continue

                parsed_entry = None

                if log_type == 'apache' or (log_type == 'auto' and 'GET' in line):
                    parsed_entry = self.parse_apache_log(line)
                elif log_type == 'nginx' or (log_type == 'auto' and 'nginx' in line):
                    parsed_entry = self.parse_nginx_log(line)
                elif log_type == 'ssh' or (log_type == 'auto' and 'ssh' in line.lower()):
                    parsed_entry = self.parse_ssh_log(line)
                elif log_type == 'firewall' or (log_type == 'auto' and '->' in line):
                    parsed_entry = self.parse_firewall_log(line)

                if parsed_entry:
                    parsed_entry['message'] = line
                    parsed_entry['hash'] = hashlib.md5(line.encode()).hexdigest()

                    threats = self.detect_threats(parsed_entry)

                    cursor.execute('''
                        INSERT OR IGNORE INTO logs
                        (timestamp, source_ip, source_port, dest_ip, dest_port, protocol, event_type, severity, message, parsed_data, hash)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ''', (
                        parsed_entry.get('timestamp', datetime.now().isoformat()),
                        parsed_entry.get('source_ip'),
                        parsed_entry.get('source_port'),
                        parsed_entry.get('dest_ip'),
                        parsed_entry.get('dest_port'),
                        parsed_entry.get('protocol'),
                        parsed_entry.get('event_type', 'unknown'),
                        'info',
                        parsed_entry.get('message'),
                        json.dumps(parsed_entry),
                        parsed_entry.get('hash')
                    ))

                    for threat in threats:
                        cursor.execute('''
                            INSERT INTO alerts
                            (timestamp, alert_type, severity, source_ip, description, evidence)
                            VALUES (?, ?, ?, ?, ?, ?)
                        ''', (
                            datetime.now().isoformat(),
                            threat['type'],
                            threat['severity'],
                            parsed_entry.get('source_ip'),
                            f"{threat['type']} detected",
                            threat['evidence']
                        ))

                    processed_count += 1

                    if processed_count % 1000 == 0:
                        conn.commit()

            conn.commit()
            conn.close()
            file_handle.close()

        except Exception as e:
            print(f"Error processing {file_path}: {e}")

        return processed_count

    def generate_dashboard_data(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        dashboard = {
            'summary': {},
            'top_threats': [],
            'timeline': [],
            'geographic_distribution': {},
            'alert_status': {}
        }

        cursor.execute('SELECT COUNT(*) FROM logs')
        dashboard['summary']['total_logs'] = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM alerts WHERE status = "open"')
        dashboard['summary']['open_alerts'] = cursor.fetchone()[0]

        cursor.execute('SELECT COUNT(*) FROM alerts WHERE severity = "critical"')
        dashboard['summary']['critical_alerts'] = cursor.fetchone()[0]

        cursor.execute('''
            SELECT alert_type, COUNT(*) as count
            FROM alerts
            WHERE timestamp > datetime('now', '-24 hours')
            GROUP BY alert_type
            ORDER BY count DESC
            LIMIT 10
        ''')

        for alert_type, count in cursor.fetchall():
            dashboard['top_threats'].append({
                'type': alert_type,
                'count': count
            })

        cursor.execute('''
            SELECT DATE(timestamp) as date, COUNT(*) as count
            FROM alerts
            WHERE timestamp > datetime('now', '-7 days')
            GROUP BY DATE(timestamp)
            ORDER BY date
        ''')

        for date, count in cursor.fetchall():
            dashboard['timeline'].append({
                'date': date,
                'count': count
            })

        cursor.execute('''
            SELECT status, COUNT(*) as count
            FROM alerts
            GROUP BY status
        ''')

        for status, count in cursor.fetchall():
            dashboard['alert_status'][status] = count

        conn.close()
        return dashboard

    def export_report(self, output_file='security_report.json'):
        dashboard_data = self.generate_dashboard_data()
        brute_force_alerts = self.analyze_brute_force()
        anomalies = self.analyze_anomalies()
        correlations = self.correlate_events()
        threat_intel = self.generate_threat_intel()

        report = {
            'timestamp': datetime.now().isoformat(),
            'dashboard': dashboard_data,
            'brute_force_alerts': brute_force_alerts,
            'anomalies': anomalies,
            'correlations': correlations,
            'threat_intelligence': threat_intel
        }

        with open(output_file, 'w') as f:
            json.dump(report, f, indent=2)

        return report

def main():
    analyzer = LogAnalyzer()

    print("Log Analysis & SIEM Tool")
    print("=" * 30)

    while True:
        print("\n1. Process log file")
        print("2. Analyze brute force attacks")
        print("3. Detect anomalies")
        print("4. Correlate events")
        print("5. Generate dashboard")
        print("6. Export report")
        print("7. Exit")

        choice = input("\nEnter your choice: ")

        if choice == '1':
            file_path = input("Enter log file path: ")
            log_type = input("Enter log type (apache/nginx/ssh/firewall/auto): ") or 'auto'

            if os.path.exists(file_path):
                count = analyzer.process_log_file(file_path, log_type)
                print(f"Processed {count} log entries")
            else:
                print("File not found!")

        elif choice == '2':
            alerts = analyzer.analyze_brute_force()
            print(f"Found {len(alerts)} brute force alerts:")
            for alert in alerts:
                print(f"- {alert['source_ip']}: {alert['attempts']} attempts ({alert['severity']})")

        elif choice == '3':
            anomalies = analyzer.analyze_anomalies()
            print(f"Found {len(anomalies)} anomalies:")
            for anomaly in anomalies:
                print(f"- {anomaly['type']}: {anomaly['description']}")

        elif choice == '4':
            correlations = analyzer.correlate_events()
            print(f"Found {len(correlations)} correlated events:")
            for correlation in correlations:
                print(f"- {correlation['source_ip']}: {correlation['description']}")

        elif choice == '5':
            dashboard = analyzer.generate_dashboard_data()
            print("Dashboard Summary:")
            print(f"Total logs: {dashboard['summary']['total_logs']}")
            print(f"Open alerts: {dashboard['summary']['open_alerts']}")
            print(f"Critical alerts: {dashboard['summary']['critical_alerts']}")
            print(f"Top threats: {len(dashboard['top_threats'])}")

        elif choice == '6':
            report = analyzer.export_report()
            print("Report exported to security_report.json")

        elif choice == '7':
            break

        else:
            print("Invalid choice!")

if __name__ == "__main__":
    main()